# 🔄 Fase 4 – Raffinamento Iterativo della Schedulazione

**SmartScheduler | Ciclo di ottimizzazione dell'equità (Fairness Loop)**

---

## Obiettivo
Migliorare **iterativamente** l'equità (fairness) della schedulazione senza violare nessuno dei vincoli rigidi (Hard Constraints).

## Architettura (Best Practice)

| Componente | Ruolo |
|---|---|
| **Dizionario `min_scores`** | Parametro mutabile che accumula le soglie minime di soddisfazione per ogni lavoratore |
| **LLM (JSON Strutturato)** | Analizza i punteggi attuali, decide la nuova soglia da alzare. NON scrive codice OR-Tools |
| **OR-Tools (ricostruito)** | Riceve i parametri aggiornati e risolve da zero ad ogni iterazione (stateless) |
| **Loop Python** | Orchestra il ciclo, cattura INFEASIBLE e salva l'ultima soluzione valida |

## Flusso per Iterazione
```
1. Solve(min_scores) → punteggi attuali
2. LLM analizza → JSON: {target_worker, new_min_score}
3. Aggiorna min_scores[target_worker] = new_min_score
4. Se INFEASIBLE → STOP, restituisci ultima soluzione valida
   Se FEASIBLE   → salva soluzione e torna a 1
```

---
## Cella 1 – Import e Configurazione Globale

In [ ]:
# ============================================================
# IMPORT
# ============================================================
import os
import json
import copy
import re
import csv
import time
import math
from datetime import date
from typing import Dict, Optional

from ortools.sat.python import cp_model
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

# Import moduli interni SmartScheduler
import input_data
from drafting_agent import load_problem_data, ProblemData, ScheduleResult, SATISFACTION_SCALE

load_dotenv()

# ============================================================
# PARAMETRI GLOBALI – FASE 4
# ============================================================

# ▸ Caso da raffinare: 'A' oppure 'B'
CASE_LABEL = "A"

# ▸ Limite massimo di iterazioni del loop (best practice #5)
MAX_ITER = 5

# ▸ Timeout per singola chiamata al solver (alzato a 120.0s)
SOLVER_TIMEOUT_SECONDS = 60.0

# ▸ Numero massimo di tentativi se l'LLM non produce JSON valido
LLM_MAX_RETRIES = 3

# ▸ Modello LLM
LLM_MODEL = "gemini-2.5-flash"

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY non trovata. "
        "Imposta la variabile nel file .env o con:\n"
        "  $env:GEMINI_API_KEY = 'la-tua-chiave'  (PowerShell)"
    )

print(f"✅ Configurazione caricata. Caso: {CASE_LABEL} | MAX_ITER: {MAX_ITER} | Modello: {LLM_MODEL}")

---
## Cella 2 – Caricamento Dati e Soluzione Iniziale (dal CSV)

Legge il CSV prodotto dalla Fase 2/3 per ottenere i punteggi di soddisfazione iniziali da cui partire.

In [ ]:
# ============================================================
# CARICAMENTO DATI DEL PROBLEMA
# ============================================================
data: ProblemData = load_problem_data(CASE_LABEL)

print(f"\n📋 Caso {CASE_LABEL} caricato:")
print(f"   Lavoratori totali  : {len(data.worker_ids)}")
print(f"   Standard           : {len(data.standard_ids)}")
print(f"   Specializzati      : {len(data.specialized_ids)}")

# ============================================================
# LETTURA DEI PUNTEGGI INIZIALI DAL CSV (output Fase 2/3)
# ============================================================
csv_path = f"schedule_case_{CASE_LABEL}.csv"

initial_scores: Dict[str, float] = {}
initial_names: Dict[str, str] = {}
initial_assignments: Dict[str, Dict[int, Optional[str]]] = {}

with open(csv_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        wid = row["worker_id"].strip()
        if wid:
            initial_scores[wid] = float(row["soddisfazione"])
            initial_names[wid] = row["nome"].strip()
            initial_assignments[wid] = {}
            for d_idx, d_date in enumerate(input_data.PLANNING_DATES):
                val = row.get(d_date.isoformat(), "-")
                initial_assignments[wid][d_idx] = val if val in input_data.SHIFT_CODES else None

print(f"\n📊 Punteggi di soddisfazione iniziali (dal CSV):")
print(f"   {'ID':<6} {'Nome':<22} {'Soddisfazione':>14}")
print(f"   {'-'*44}")
for wid in sorted(initial_scores):
    marker = " ← MINIMO" if initial_scores[wid] == min(initial_scores.values()) else ""
    print(f"   {wid:<6} {initial_names[wid]:<22} {initial_scores[wid]:>14.1f}{marker}")

worst_worker = min(initial_scores, key=initial_scores.get)
print(f"\n⚠️  Lavoratore meno soddisfatto: {worst_worker} ({initial_names[worst_worker]}) "
      f"— score: {initial_scores[worst_worker]}")


---
## Cella 3 – Builder Deterministico (Stateless) con Soglie Parametrizzate

**Best Practice #1 e #3:** Il modello OR-Tools viene ricostruito da zero ad ogni iterazione.
Il codice Python rimane statico; i parametri `min_scores` vengono iniettati deterministicamente.

In [ ]:
def build_and_solve_with_floors(
    data: ProblemData,
    min_scores: Dict[str, float],
    max_time_seconds: float = 30.0,
    hints: Optional[Dict[str, Dict[int, Optional[str]]]] = None,
) -> ScheduleResult:
    """
    Ricostruisce il modello CP-SAT da zero (stateless) iniettando le soglie
    minime di soddisfazione come vincoli addizionali.

    Best Practice #1: Il codice è statico. Solo min_scores cambia ad ogni iterazione.
    Best Practice #3: Il modello viene ricreato da zero → zero 'vincoli fantasma'.
    Best Practice #5: max_time_seconds limita il tempo del solver per singola iterazione.

    Parameters
    ----------
    data            : ProblemData  – dati del problema (lavoratori, preferenze, staffing)
    min_scores      : dict         – {worker_id: soglia_minima_soddisfazione}
    max_time_seconds: float        – timeout OR-Tools per singola esecuzione

    Returns
    -------
    ScheduleResult con feasible=True/False e satisfaction_per_worker compilato.
    """
    model = cp_model.CpModel()
    num_days    = input_data.NUM_DAYS
    shift_codes = input_data.SHIFT_CODES   # ["M", "P", "N"]
    shifts      = input_data.SHIFTS
    hc          = input_data.HARD_CONSTRAINTS

    shift_hours  = {s: shifts[s]["durata_ore"]  for s in shift_codes}  # M/P=6, N=12
    shift_weight = {s: shifts[s]["peso_turni"]  for s in shift_codes}  # M/P=1, N=2

    # ------------------------------------------------------------------
    # VARIABILI DECISIONALI: x[(w, d, s)] ∈ {0, 1}
    # ------------------------------------------------------------------
    x = {
        (w, d, s): model.NewBoolVar(f"x_{w}_{d}_{s}")
        for w in data.worker_ids
        for d in range(num_days)
        for s in shift_codes
    }

    # ------------------------------------------------------------------
    # VINCOLI HARD (invariati, copiati 1:1 da drafting_agent.py)
    # ------------------------------------------------------------------

    # HC#3 (parte 1): max 1 turno al giorno
    for w in data.worker_ids:
        for d in range(num_days):
            model.Add(sum(x[(w, d, s)] for s in shift_codes) <= hc["max_turni_per_giorno"])

    # HC#2: esattamente 25 turni al mese (Notte vale 2)
    for w in data.worker_ids:
        model.Add(
            sum(shift_weight[s] * x[(w, d, s)]
                for d in range(num_days) for s in shift_codes)
            == hc["turni_mensili_esatti"]
        )

    # HC#1 + HC#6: max 36 ore settimanali + almeno 1 riposo/settimana
    week = 7
    for w in data.worker_ids:
        for t in range(0, num_days - week + 1):
            giorni_finestra = range(t, t + week)
            model.Add(
                sum(shift_hours[s] * x[(w, d, s)]
                    for d in giorni_finestra for s in shift_codes)
                <= hc["max_ore_settimanali"]
            )
            model.Add(
                sum(x[(w, d, s)] for d in giorni_finestra for s in shift_codes)
                <= week - hc["giorni_riposo_minimi"]
            )

    # HC#5: 2 giorni liberi obbligatori dopo ogni Notte
    riposi = hc["riposi_obbligatori_dopo_notte"]
    for w in data.worker_ids:
        for d in range(num_days):
            for k in range(1, riposi + 1):
                if d + k < num_days:
                    for s in shift_codes:
                        model.Add(x[(w, d, "N")] + x[(w, d + k, s)] <= 1)

    # HC#3 (parte 2): divieto Notte(d) → Mattina(d+1)
    for w in data.worker_ids:
        for d in range(num_days - 1):
            model.Add(x[(w, d, "N")] + x[(w, d + 1, "M")] <= 1)

    # Indisponibilità (da preferenze Fase 1)
    for w in data.worker_ids:
        for d in data.unavailable.get(w, set()):
            for s in shift_codes:
                model.Add(x[(w, d, s)] == 0)

    # Staffing (dipende dal caso)
    if data.case_label == "A":
        min_per_turno = data.staffing["min_lavoratori_per_turno"]
        for d in range(num_days):
            for s in shift_codes:
                model.Add(sum(x[(w, d, s)] for w in data.worker_ids) >= min_per_turno)
    else:
        min_std  = data.staffing["min_standard_per_turno"]
        min_spec = data.staffing["min_specializzati_per_turno"]
        for d in range(num_days):
            for s in shift_codes:
                model.Add(sum(x[(w, d, s)] for w in data.standard_ids)    >= min_std)
                model.Add(sum(x[(w, d, s)] for w in data.specialized_ids) >= min_spec)

    # ------------------------------------------------------------------
    # VINCOLI SOFT FLOOR (FASE 4) – PARAMETRIZZAZIONE A DIZIONARIO
    # Best Practice #1: iniettati come parametri, il codice OR-Tools non cambia mai.
    # ------------------------------------------------------------------
    # Costruiamo variabili IntVar per la soddisfazione per poter applicare
    # i vincoli di soglia in modo deterministico.
    satisfaction_vars: Dict[str, cp_model.IntVar] = {}
    for w in data.worker_ids:
        pesi = data.preferences[w]["satisfaction_weights"]
        # Calcola i limiti teorici per definire il dominio dell'IntVar
        min_pos_coef = sum(
            int(round(pesi[s] * SATISFACTION_SCALE))
            for d in range(num_days) for s in shift_codes
            if int(round(pesi[s] * SATISFACTION_SCALE)) > 0
        )
        max_neg_coef = sum(
            int(round(pesi[s] * SATISFACTION_SCALE))
            for d in range(num_days) for s in shift_codes
            if int(round(pesi[s] * SATISFACTION_SCALE)) < 0
        )
        satisfaction_vars[w] = model.NewIntVar(
            max_neg_coef,
            min_pos_coef,
            f"sat_{w}"
        )
        model.Add(
            satisfaction_vars[w] == sum(
                int(round(pesi[s] * SATISFACTION_SCALE)) * x[(w, d, s)]
                for d in range(num_days)
                for s in shift_codes
            )
        )

        # Applica la soglia minima per questo lavoratore (il cuore della Fase 4)
        raw_floor = min_scores.get(w, float("-inf"))
        if math.isfinite(raw_floor):
            floor_scaled = int(round(raw_floor * SATISFACTION_SCALE))
            if floor_scaled > max_neg_coef:
                model.Add(satisfaction_vars[w] >= floor_scaled)


    # Add Hints if provided
    if hints:
        for w in data.worker_ids:
            for d in range(num_days):
                for s in shift_codes:
                    if hints.get(w, {}).get(d) == s:
                        model.AddHint(x[(w, d, s)], 1)
                    else:
                        model.AddHint(x[(w, d, s)], 0)

    # ------------------------------------------------------------------
    # FUNZIONE OBIETTIVO: massimizza soddisfazione totale

    # ------------------------------------------------------------------
    model.Maximize(sum(satisfaction_vars.values()))

    # ------------------------------------------------------------------
    # RISOLUZIONE
    # ------------------------------------------------------------------
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds   = max_time_seconds  # Best Practice #5
    solver.parameters.num_search_workers    = 8
    solver.parameters.log_search_progress   = False

    status      = solver.Solve(model)
    status_name = solver.StatusName(status)
    feasible    = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)

    result = ScheduleResult(
        case_label   = data.case_label,
        status_name  = status_name,
        feasible     = feasible,
        source       = "phase4_deterministic",
    )

    if not feasible:
        return result

    # Estrae la schedulazione e calcola i punteggi reali (non scalati)
    schedule: Dict[str, Dict[int, Optional[str]]] = {}
    for w in data.worker_ids:
        schedule[w] = {}
        for d in range(num_days):
            assegnato = None
            for s in shift_codes:
                if solver.Value(x[(w, d, s)]) == 1:
                    assegnato = s
                    break
            schedule[w][d] = assegnato

    result.schedule        = schedule
    result.objective_value = solver.ObjectiveValue() / SATISFACTION_SCALE

    for w in data.worker_ids:
        pesi = data.preferences[w]["satisfaction_weights"]
        result.satisfaction_per_worker[w] = round(
            sum(pesi[s] for d in range(num_days)
                for s in shift_codes if schedule[w][d] == s), 2
        )

    return result



print("✅ Funzione `build_and_solve_with_floors` definita.")


---
## Cella 4 – Interfaccia LLM con Structured Output (JSON)

**Best Practice #2:** L'LLM non scrive mai codice. Restituisce SOLO un JSON con la soglia da alzare.

In [ ]:
# Inizializzazione client LLM
llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    google_api_key=GEMINI_API_KEY,
)


def build_refinement_prompt(
    data: ProblemData,
    current_scores: Dict[str, float],
    current_min_scores: Dict[str, float],
    iteration: int,
) -> str:
    """
    Costruisce il prompt da inviare all'LLM per decidere la prossima soglia.
    L'LLM NON scrive codice: il suo unico compito è restituire un JSON.
    """
    # Ordina i lavoratori per punteggio crescente (il peggiore in testa)
    workers_sorted = sorted(current_scores.items(), key=lambda kv: kv[1])
    ranking_str = "\n".join(
        f"  {wid} ({data.worker_names[wid]}): soddisfazione={score:.1f}, "
        f"soglia_minima_attuale={current_min_scores.get(wid, 'nessuna')}"
        for wid, score in workers_sorted
    )

    worst_wid   = workers_sorted[0][0]
    worst_score = workers_sorted[0][1]

    prompt = f"""Sei il "Fairness Optimizer" di SmartScheduler (Fase 4, iterazione {iteration}).

Il tuo compito è analizzare i punteggi di soddisfazione dei lavoratori e decidere
quale soglia minima alzare per migliorare l'equità della schedulazione.

### PUNTEGGI ATTUALI (ordinati dal meno al più soddisfatto)
{ranking_str}

### REGOLE DA RISPETTARE
1. Individua il lavoratore con il punteggio più basso: è {worst_wid} ({data.worker_names[worst_wid]}) con score {worst_score:.1f}.
2. Proponi di alzare la sua soglia minima a un valore MAGGIORE del suo PUNTEGGIO ATTUALE (punteggio_attuale + step).
3. Lo step deve essere conservativo (piccolo incremento, suggerito: la differenza media
   tra il punteggio del lavoratore peggiore e il successivo, divisa per 2).
4. NON peggiorare le soglie degli altri lavoratori.
5. La nuova soglia DEVE superare strettamente il punteggio attuale del lavoratore target
   (altrimenti il solver restituirà la stessa identica soluzione dell'iterazione precedente).

### OUTPUT RICHIESTO
Rispondi SOLO con un oggetto JSON valido, racchiuso tra ```json e ```. Nessun altro testo.
Schema:
{{
  "target_worker": "<worker_id>",
  "new_min_score": <float>,
  "reasoning": "<breve spiegazione in italiano, max 2 righe>"
}}"""
    return prompt


def ask_llm_for_new_threshold(
    data: ProblemData,
    current_scores: Dict[str, float],
    current_min_scores: Dict[str, float],
    iteration: int,
    max_retries: int = LLM_MAX_RETRIES,
) -> Optional[dict]:
    """
    Chiama l'LLM e ottiene il JSON con la nuova soglia da applicare.
    Best Practice #2: L'LLM restituisce SOLO JSON strutturato, mai codice Python.

    Returns
    -------
    dict con 'target_worker', 'new_min_score', 'reasoning'
    oppure None se l'LLM non produce JSON valido dopo max_retries tentativi.
    """
    prompt = build_refinement_prompt(data, current_scores, current_min_scores, iteration)

    for attempt in range(1, max_retries + 1):
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            raw_text = response.content
            if isinstance(raw_text, list):
                raw_text = "".join(
                    b.get("text", "") if isinstance(b, dict) else str(b)
                    for b in raw_text
                )

            # Estrai il blocco JSON
            match = re.search(r"```json\s*(.*?)\s*```", raw_text, re.DOTALL)
            if not match:
                # Prova anche senza delimitatori (solo oggetto JSON grezzo)
                match = re.search(r"(\{.*?\})", raw_text, re.DOTALL)
            if not match:
                print(f"   ⚠️  Tentativo {attempt}/{max_retries}: LLM non ha restituito JSON valido.")
                continue

            payload = json.loads(match.group(1))

            # Validazione minima del payload
            if "target_worker" not in payload or "new_min_score" not in payload:
                print(f"   ⚠️  Tentativo {attempt}/{max_retries}: JSON incompleto: {payload}")
                continue

            return payload

        except (json.JSONDecodeError, AttributeError) as e:
            print(f"   ⚠️  Tentativo {attempt}/{max_retries}: errore parsing JSON: {e}")
            time.sleep(2)
        except Exception as e:
            err = str(e)
            if "429" in err or "RESOURCE_EXHAUSTED" in err:
                wait = 60
                print(f"   ⚠️  Rate limit API. Attendo {wait}s...")
                time.sleep(wait)
            else:
                print(f"   ❌ Errore LLM inatteso: {e}")
                return None

    return None


print("✅ Interfaccia LLM (JSON Structured Output) definita.")

---
## Cella 5 – Funzioni di Supporto (Logging e Salvataggio CSV)

In [ ]:
def print_iteration_summary(
    iteration: int,
    scores: Dict[str, float],
    min_scores: Dict[str, float],
    data: ProblemData,
    llm_decision: Optional[dict] = None,
) -> None:
    """Stampa un riepilogo compatto di ogni iterazione."""
    worst_wid   = min(scores, key=scores.get)
    best_wid    = max(scores, key=scores.get)
    avg_score   = sum(scores.values()) / len(scores)
    min_score   = scores[worst_wid]

    print(f"\n{'─'*60}")
    print(f"📍 ITERAZIONE {iteration}")
    print(f"   Soddisfazione min   : {min_score:.1f}  ({worst_wid} – {data.worker_names[worst_wid]})")
    print(f"   Soddisfazione max   : {scores[best_wid]:.1f}  ({best_wid} – {data.worker_names[best_wid]})")
    print(f"   Soddisfazione media : {avg_score:.1f}")
    print(f"   Gap min/max         : {scores[best_wid] - min_score:.1f}")

    if llm_decision:
        tw  = llm_decision["target_worker"]
        nms = llm_decision["new_min_score"]
        rsn = llm_decision.get("reasoning", "–")
        print(f"   🤖 Decisione LLM    : alzare soglia di {tw} "
              f"({data.worker_names.get(tw, '?')}) a {nms}")
        print(f"   📝 Motivazione      : {rsn}")


def save_result_to_csv(result: ScheduleResult, data: ProblemData, suffix: str = "fase4") -> str:
    """
    Salva la schedulazione raffinata in un file CSV nello stesso formato
    della Fase 2 (schedule_case_A.csv / schedule_case_B.csv).

    Il file viene nominato: schedule_case_{CASE}_{suffix}.csv
    """
    dates       = input_data.PLANNING_DATES
    out_path    = f"schedule_case_{result.case_label}_{suffix}.csv"
    shift_codes = input_data.SHIFT_CODES
    shifts      = input_data.SHIFTS

    shift_weight = {s: shifts[s]["peso_turni"] for s in shift_codes}

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        date_headers = [d.isoformat() for d in dates]
        fieldnames   = ["worker_id", "nome"] + date_headers + ["tot_turni_pesati", "soddisfazione"]
        writer       = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for wid in data.worker_ids:
            row           = {"worker_id": wid, "nome": data.worker_names[wid]}
            tot_pesati    = 0
            for d_idx, d_date in enumerate(dates):
                turno = result.schedule.get(wid, {}).get(d_idx)
                row[d_date.isoformat()] = turno if turno else "-"
                if turno:
                    tot_pesati += shift_weight[turno]

            row["tot_turni_pesati"] = tot_pesati
            row["soddisfazione"]    = result.satisfaction_per_worker.get(wid, 0)
            writer.writerow(row)

    print(f"   💾 Soluzione salvata in: {out_path}")
    return out_path


print("✅ Funzioni di logging e salvataggio CSV definite.")

---
## Cella 6 – 🚀 Loop Principale della Fase 4

Il cuore dell'architettura: **Python orchestra, LLM decide la strategia, OR-Tools risolve.**

In [ ]:
# ============================================================
# INIZIALIZZAZIONE DEL CICLO
# ============================================================

# Dizionario delle soglie minime (Best Practice #1)
# Inizialmente a -∞ per tutti → nessun vincolo aggiuntivo rispetto alla Fase 2
min_scores: Dict[str, float] = {wid: float("-inf") for wid in data.worker_ids}

# Stato del loop
best_result:  Optional[ScheduleResult] = None   # Ultima soluzione FEASIBLE
best_scores:  Dict[str, float]         = {}      # Punteggi dell'ultima FEASIBLE
history:      list                     = []      # Log di ogni iterazione

print("=" * 60)
print(f"🔄 FASE 4 – LOOP DI RAFFINAMENTO | Caso {CASE_LABEL}")
print(f"   MAX_ITER: {MAX_ITER} | Timeout solver: {SOLVER_TIMEOUT_SECONDS}s")
print("=" * 60)

# ============================================================
# ITERAZIONE 0: Risoluzione base (senza floor aggiuntivi)
# Verifica che la schedulazione iniziale sia ancora producibile
# ============================================================
print("\n⚙️  Iterazione 0 – Risoluzione base (verifica fattibilità iniziale)...")
base_result = build_and_solve_with_floors(data, min_scores, SOLVER_TIMEOUT_SECONDS, initial_assignments)

if not base_result.feasible:
    raise RuntimeError(
        f"❌ CRITICO: La schedulazione base è INFEASIBLE per il Caso {CASE_LABEL}.\n"
        "   Verificare i vincoli hard in drafting_agent.py."
    )

best_result = base_result
best_scores = dict(base_result.satisfaction_per_worker)
print_iteration_summary(0, best_scores, min_scores, data)
history.append({"iteration": 0, "scores": dict(best_scores), "min_scores": dict(min_scores), "llm_decision": None, "status": "FEASIBLE"})

# ============================================================
# LOOP PRINCIPALE (Best Practice #5: limite MAX_ITER)
# ============================================================
for iteration in range(1, MAX_ITER + 1):
    print(f"\n{'='*60}")
    print(f"🔄 Iterazione {iteration}/{MAX_ITER}")

    # ----------------------------------------------------------
    # STEP A: LLM decide la nuova soglia (Best Practice #2)
    # ----------------------------------------------------------
    print(f"   🤖 Consultazione LLM per la nuova soglia...")
    llm_decision = ask_llm_for_new_threshold(
        data            = data,
        current_scores  = best_scores,
        current_min_scores = min_scores,
        iteration       = iteration,
    )

    if llm_decision is None:
        print(f"   ❌ LLM non ha prodotto un JSON valido dopo {LLM_MAX_RETRIES} tentativi. Interruzione.")
        break

    target_wid    = llm_decision["target_worker"]
    new_min_score = float(llm_decision["new_min_score"])

    # Sicurezza: il target deve essere un worker valido
    if target_wid not in data.worker_ids:
        print(f"   ⚠️  LLM ha proposto worker sconosciuto '{target_wid}'. Interruzione.")
        break

    # Sicurezza: la nuova soglia non deve essere inferiore a quella attuale
    current_floor = min_scores.get(target_wid, float("-inf"))
    if new_min_score <= best_scores[target_wid]:
        new_min_score = best_scores[target_wid] + 2.0
        print(f"   ⚠️  Soglia LLM troppo bassa. Forzata a {new_min_score} per imporre un miglioramento.")
    if False:
        print(f"   ⚠️  La nuova soglia ({new_min_score}) non migliora quella attuale ({current_floor}). "
              "Convergenza raggiunta.")
        break

    # ----------------------------------------------------------
    # STEP B: Prepara i nuovi floor (Regola di Robin Hood)
    # ----------------------------------------------------------
    new_min_scores = dict(min_scores)
    new_min_scores[target_wid] = new_min_score

    # Il livello minimo tra "gli altri" (escludendo il target)
    altri_scores = [best_scores[w] for w in data.worker_ids if w != target_wid]
    min_other_score = min(altri_scores) if altri_scores else float("-inf")

    # Regola Robin Hood: puoi togliere punti a chi sta bene, ma nessuno 
    # può scendere sotto la soglia di chi sta peggio tra "gli altri"
    for wid in data.worker_ids:
        if wid != target_wid:
            new_min_scores[wid] = max(
                min_scores.get(wid, float("-inf")), 
                min_other_score
            )

    # ----------------------------------------------------------
    # STEP C: Ricostruisci il modello da zero con i nuovi floor
    # Best Practice #3: stateless, zero vincoli fantasma
    # ----------------------------------------------------------
    print(f"   ⚙️  Esecuzione solver (timeout: {SOLVER_TIMEOUT_SECONDS}s)...")
    result = build_and_solve_with_floors(data, new_min_scores, SOLVER_TIMEOUT_SECONDS, best_result.schedule)

    print(f"   📊 Status solver: {result.status_name}")

    # ----------------------------------------------------------
    # STEP D: Gestione INFEASIBLE (Best Practice #4)
    # ----------------------------------------------------------
    if not result.feasible:
        print(f"\n🏁 STOP: Ottimo di Pareto raggiunto alla iterazione {iteration}.")
        print(f"   Il solver è INFEASIBLE con la soglia {new_min_score} per {target_wid}.")
        print(f"   → La soluzione migliore è quella dell'iterazione precedente.")
        history.append({
            "iteration": iteration,
            "scores": None,
            "min_scores": new_min_scores,
            "llm_decision": llm_decision,
            "status": "INFEASIBLE"
        })
        break

    # ----------------------------------------------------------
    # STEP E: Aggiorna lo stato (soluzione migliorata!)
    # ----------------------------------------------------------
    min_scores  = new_min_scores
    best_result = result
    best_scores = dict(result.satisfaction_per_worker)

    print_iteration_summary(iteration, best_scores, min_scores, data, llm_decision)
    history.append({
        "iteration": iteration,
        "scores": dict(best_scores),
        "min_scores": dict(min_scores),
        "llm_decision": llm_decision,
        "status": result.status_name
    })

else:
    print(f"\n⚠️  Raggiunto il limite massimo di {MAX_ITER} iterazioni.")

# ============================================================
# RIEPILOGO FINALE
# ============================================================
print(f"\n{'='*60}")
print(f"✅ FASE 4 COMPLETATA – Caso {CASE_LABEL}")
print(f"   Iterazioni eseguite: {len(history) - 1}")
print(f"   Soddisfazione totale: {sum(best_scores.values()):.1f}")
print(f"   Soddisfazione min   : {min(best_scores.values()):.1f}")
print(f"   Soddisfazione media : {sum(best_scores.values())/len(best_scores):.1f}")
print(f"{'='*60}")


---
## Cella 7 – Analisi Comparativa (Prima vs Dopo)

Confronta i punteggi iniziali (dal CSV della Fase 2) con quelli raffinati dalla Fase 4.

In [ ]:
import pandas as pd

# Costruisci tabella comparativa
rows = []
for wid in data.worker_ids:
    score_before = initial_scores.get(wid, float("nan"))
    score_after  = best_scores.get(wid, float("nan"))
    delta        = score_after - score_before
    rows.append({
        "ID": wid,
        "Nome": data.worker_names[wid],
        "Soddisfazione Iniziale": score_before,
        "Soddisfazione Fase 4": score_after,
        "Δ (miglioramento)": delta,
        "Soglia Min Applicata": min_scores.get(wid, float("-inf")),
    })

df = pd.DataFrame(rows).set_index("ID")
df_sorted = df.sort_values("Soddisfazione Iniziale")

print("\n📊 CONFRONTO SODDISFAZIONE: Prima vs Dopo (ordinato per soddisfazione iniziale crescente)\n")
print(df_sorted.to_string(float_format="{:.1f}".format))

print(f"\n📈 Riepilogo aggregato:")
print(f"   Soddisfazione totale  PRIMA: {df['Soddisfazione Iniziale'].sum():.1f}")
print(f"   Soddisfazione totale  DOPO : {df['Soddisfazione Fase 4'].sum():.1f}")
print(f"   Soddisfazione min     PRIMA: {df['Soddisfazione Iniziale'].min():.1f}")
print(f"   Soddisfazione min     DOPO : {df['Soddisfazione Fase 4'].min():.1f}")
print(f"   Gap min/max           PRIMA: {df['Soddisfazione Iniziale'].max() - df['Soddisfazione Iniziale'].min():.1f}")
print(f"   Gap min/max           DOPO : {df['Soddisfazione Fase 4'].max() - df['Soddisfazione Fase 4'].min():.1f}")

---
## Cella 8 – Salvataggio CSV della Schedulazione Raffinata

In [ ]:
# Salva il CSV raffinato
if best_result and best_result.feasible:
    output_path = save_result_to_csv(best_result, data, suffix="fase4")
    print(f"\n✅ Schedulazione raffinata salvata in: {output_path}")
    print("   Puoi aprirlo con Excel / Pandas per confrontarlo con la versione originale.")
else:
    print("❌ Nessuna soluzione feasible da salvare.")

---
## Cella 9 – Log Completo della Cronologia Iterazioni

In [ ]:
print("\n📜 CRONOLOGIA COMPLETA DELLE ITERAZIONI\n")
for entry in history:
    it  = entry["iteration"]
    st  = entry["status"]
    dec = entry.get("llm_decision")
    sc  = entry.get("scores") or {}

    print(f"  Iter {it:>2d} | Status: {st:<12}", end="")
    if dec:
        print(f"| LLM: alza soglia {dec['target_worker']} → {dec['new_min_score']}", end="")
    if sc:
        worst_s = min(sc.values())
        avg_s   = sum(sc.values()) / len(sc)
        print(f"  | min={worst_s:.1f}  avg={avg_s:.1f}", end="")
    print()